# Global Video Game Sales

***Main Task:*** Build an interactive dashboard to explore & identify key insights in the global video games industry

## Project Brief

**The situation**:  I am hired as a Data Analyst for Mavendo Games, a gaming Company known for numerous worldwide hits.

**The Assignment:** Senior Leadership at Mavendo games wants more visibility into the gaming
                    industry, so they can determine which games and markets are most promising.
                    You've been asked to produce an interactive dashboard that enables the
                    leadership team to explore things like the most popular genres, titles,
                    consoles, and more. 

## Objective 1: Profile & QA the data

- Import the vgchartz-2024.csv file
- Check that each column has an appropriate data type
- Rename the "title", "genre", "publisher", and "developer" columns to start with capital letters
- Create a "release_year" column based on the "release_date" column


In [35]:
# ============================================================================
# IMPORT LIBRARIES
# ============================================================================
from dash import Dash, dcc, html, dash_table
import dash_bootstrap_components as dbc
from dash.dependencies import Output, Input
from dash.exceptions import PreventUpdate
from dash_bootstrap_templates import load_figure_template

import plotly.express as px
import pandas as pd
import numpy as np

In [36]:
# ============================================================================
# DATA IMPORT

# ============================================================================

video_games = pd.read_csv(r"C:\Users\MilosIlic\OneDrive - Valcon Business Development A S\Data Plaftom (Power BI)\Python\Python_Exercise_Projects\Global Video Game Sales\Video+Game+Sales\vgchartz-2024.csv")


In [37]:
# Display first 5 rows to preview the dataset structure and content
video_games.head()

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,NaN
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,NaN
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17,NaN
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2018-01-14


In [38]:
# ============================================================================
# DATA PROFILING

# ============================================================================

video_games.info()

# Display comprehensive dataframe information:# - Memory usage

# - Column names and count# - Non-null value counts (identify missing data)
# - Data types for each column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  object 
 1   title         64016 non-null  object 
 2   console       64016 non-null  object 
 3   genre         64016 non-null  object 
 4   publisher     64016 non-null  object 
 5   developer     63999 non-null  object 
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  object 
 13  last_update   17879 non-null  object 
dtypes: float64(6), object(8)
memory usage: 6.8+ MB


In [40]:
# ============================================================================
# DATA TYPE CONVERSION
# ============================================================================
# Convert date columns from object (string) to datetime format
# This enables date-based operations, filtering, and time series analysis
# errors='coerce' converts invalid dates to NaT (Not a Time) instead of raising errors

video_games['release_date'] = pd.to_datetime(video_games['release_date'], errors='coerce')
video_games['last_update'] = pd.to_datetime(video_games['last_update'], errors='coerce')

In [41]:
# ============================================================================
# DATA TRANSFORMATION
# ============================================================================
# Rename key columns to follow title case convention for better readability
# This improves dashboard presentation and maintains naming consistency
video_games = video_games.rename(columns={
    'title': 'Title',
    'genre': 'Genre',
    'publisher': 'Publisher',
    'developer': 'Developer'
})

# Extract year from release_date to create new column for time-based analysis

# Using float to handle null values (year extraction from NaT returns NaN)video_games.head()

video_games['release_year'] = video_games['release_date'].dt.year.astype(float)# Display updated dataframe structure


In [43]:
# Generate statistical summary of numerical columns

# Includes count, mean, std dev, min, quartiles, and max values
video_games.describe()

,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,release_year
count,6678.000000,18922.000000,12637.000000,6726.000000,12824.000000,15128.000000,56965,17879,56965.000000
mean,7.220440,0.349113,0.264740,0.102281,0.149472,0.043041,2006-11-14 06:33:03.491617792,2020-01-11 00:45:49.683986944,2006.359572
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1971-12-03 00:00:00,2017-11-28 00:00:00,1971.000000
25%,6.400000,0.030000,0.050000,0.020000,0.010000,0.000000,2001-03-28 00:00:00,2018-08-08 00:00:00,2001.000000
50%,7.500000,0.120000,0.120000,0.040000,0.040000,0.010000,2008-09-16 00:00:00,2019-04-21 00:00:00,2008.000000
75%,8.300000,0.340000,0.280000,0.120000,0.140000,0.030000,2012-12-27 00:00:00,2021-03-30 00:00:00,2012.000000
max,10.000000,20.320000,9.760000,2.130000,9.850000,3.120000,2024-12-31 00:00:00,2024-01-28 00:00:00,2024.000000
std,1.457066,0.807462,0.494787,0.168811,0.392653,0.126643,NaN,NaN,8.617813


## Objective 2: Prepare the data for visualization

- Create an "annual sales" table that calculates the sum of total sales by year
- Create a "top 10 titles" table which contains the ten highest selling titles by "total_sales", ranked from highest to lowest

In [44]:
# ============================================================================
# DATA AGGREGATION - ANNUAL SALES

# ==========================================================================

# Group data by release year and calculate total sales for each year# Display the most recent years

# This creates a time series dataset for trend analysis
annual_sales = video_games.groupby('release_year')['total_sales'].sum().reset_index()
annual_sales.tail()

,release_year,total_sales
46,2020.0,3.45
47,2021.0,0.00
48,2022.0,0.00
49,2023.0,0.00
50,2024.0,0.00


In [45]:
# ============================================================================
# DATA AGGREGATION - TOP 10 TITLES
# ============================================================================
# Calculate total sales for each game title across all platforms and regions
# Sort in descending order and select top 10 best-selling titles
top_10_titles = (video_games.groupby('Title')['total_sales']
                 .sum()
                 .reset_index()
                 .sort_values(by='total_sales', ascending=False)
                 .head(10))

# Display the top performers
top_10_titles

,Title,total_sales
13724,Grand Theft Auto V,64.29
5266,Call of Duty: Black Ops,30.99
5281,Call of Duty: Modern Warfare 3,30.71
5273,Call of Duty: Black Ops II,29.59
5277,Call of Duty: Ghosts,28.80
5271,Call of Duty: Black Ops 3,26.72
5280,Call of Duty: Modern Warfare 2,25.02
20998,Minecraft,24.01
13719,Grand Theft Auto IV,22.53
5265,Call of Duty: Advanced Warfare,21.78


## Objective 3: Build an interactive dashboard

- Create a line chart to plot total sales by year
- Create a bar chart of the top ten selling titles of all time
- Add the two charts to a dashboard that allows the user to select "title", "genre", "publisher", "developer", and "console" with a dropdown, and "total_sales", "jp_sales", "na_sales", "pal_sales" and "other_sales" with radio buttons
- Use the dropdown menu to control the category labels in the bar chart and the radio buttons to control the sales figures shown both charts

In [46]:
# ============================================================================
# STATIC VISUALIZATION - LINE CHART
# ============================================================================
# Create line chart showing sales trends over time
# This provides quick preview before building interactive dashboard
px.line(annual_sales, x='release_year', y='total_sales', title='Annual Global Video Game Sales')

In [47]:
# ============================================================================
# STATIC VISUALIZATION - BAR CHART
# ============================================================================
# Create bar chart displaying top 10 best-selling video game titles
# This provides quick preview before building interactive dashboard
px.bar(
    top_10_titles,
    x='Title',
    y='total_sales',
    height=600,
    title='Top 10 Video Game Titles by Total Sales')

In [48]:
# ============================================================================
# INTERACTIVE DASHBOARD APPLICATION
# ============================================================================
# Initialize Dash app with Slate theme (dark, professional appearance)
app = Dash(__name__, external_stylesheets=[dbc.themes.SLATE])

# ============================================================================
# DASHBOARD LAYOUT
# ============================================================================
app.layout = dbc.Container([
    # Main title
    html.H1("Global Video Game Sales Dashboard", className="text-center my-4"),
    
    # ========================================================================
    # CONTROL PANEL ROW
    # ========================================================================
    dbc.Row([
        # Left column: Category dropdown
        dbc.Col([
            html.Label("Select Category:", className="fw-bold"),
            dcc.Dropdown(
                id='category-dropdown',
                options=[
                    {'label': 'Title', 'value': 'Title'},
                    {'label': 'Genre', 'value': 'Genre'},
                    {'label': 'Publisher', 'value': 'Publisher'},
                    {'label': 'Developer', 'value': 'Developer'},
                    {'label': 'Console', 'value': 'console'}
                ],
                value='Title',
                clearable=False
            )
        ], width=6),
        
        # Right column: Sales region radio buttons
        dbc.Col([
            html.Label("Select Sales Region:", className="fw-bold"),
            dcc.RadioItems(
                id='sales-radio',
                options=[
                    {'label': ' Total Sales', 'value': 'total_sales'},
                    {'label': ' Japan Sales', 'value': 'jp_sales'},
                    {'label': ' North America Sales', 'value': 'na_sales'},
                    {'label': ' PAL Region Sales', 'value': 'pal_sales'},
                    {'label': ' Other Sales', 'value': 'other_sales'}
                ],
                value='total_sales',
                labelStyle={'display': 'block', 'margin-bottom': '8px'}
            )
        ], width=6)
    ], className="mb-4"),
    
    # ========================================================================
    # LINE CHART ROW
    # ========================================================================
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='line-chart')
        ], width=12)
    ], className="mb-4"),
    
    # ========================================================================
    # BAR CHART ROW
    # ========================================================================
    dbc.Row([
        dbc.Col([
            dcc.Graph(id='bar-chart')
        ], width=12)
    ])
], fluid=True)

# ============================================================================
# CALLBACK FUNCTION - DYNAMIC CHART UPDATES
# ============================================================================
# This function updates both charts when user changes dropdown or radio selection
@app.callback(
    [Output('line-chart', 'figure'),
     Output('bar-chart', 'figure')],
    [Input('category-dropdown', 'value'),
     Input('sales-radio', 'value')]
)
def update_charts(category, sales_column):
    """
    Update dashboard charts based on user selections.
    
    Parameters:
    -----------
    category : str
        Selected category (Title, Genre, Publisher, Developer, or Console)
    sales_column : str
        Selected sales metric (total_sales, jp_sales, na_sales, pal_sales, other_sales)
    
    Returns:
    --------
    tuple
        (line_fig, bar_fig) - Updated Plotly figure objects
    """
    
    # ========================================================================
    # DATA PREPARATION
    # ========================================================================
    # Aggregate sales by year for line chart
    annual_data = video_games.groupby('release_year')[sales_column].sum().reset_index()
    
    # Aggregate and rank top 10 items in selected category for bar chart
    top_10_data = (video_games.groupby(category)[sales_column]
                   .sum()
                   .reset_index()
                   .sort_values(by=sales_column, ascending=False)
                   .head(10))
    
    # Human-readable labels for sales metrics
    sales_labels = {
        'total_sales': 'Total Sales (millions)',
        'jp_sales': 'Japan Sales (millions)',
        'na_sales': 'North America Sales (millions)',
        'pal_sales': 'PAL Region Sales (millions)',
        'other_sales': 'Other Sales (millions)'
    }
    
    # ========================================================================
    # LINE CHART CREATION
    # ========================================================================
    # Create line chart showing sales trends over time
    line_fig = px.line(
        annual_data,
        x='release_year',
        y=sales_column,
        title=f'Annual Video Game Sales - {sales_labels[sales_column]}',
        labels={'release_year': 'Release Year', sales_column: sales_labels[sales_column]}
    )
    
    # Customize line appearance
    line_fig.update_traces(line_color='#00d9ff', line_width=3)
    
    # Apply dark theme styling to match Slate theme
    line_fig.update_layout(
        template='plotly_dark',
        paper_bgcolor='#272b30',
        plot_bgcolor='#272b30',
        font=dict(color='#ffffff')
    )
    
    # ========================================================================
    # BAR CHART CREATION
    # ========================================================================
    # Create bar chart showing top 10 items in selected category
    bar_fig = px.bar(
        top_10_data,
        x=category,
        y=sales_column,
        title=f'Top 10 {category} by {sales_labels[sales_column]}',
        labels={category: category, sales_column: sales_labels[sales_column]},
        height=500
    )
    
    # Customize bar appearance
    bar_fig.update_traces(marker_color='#00d9ff')
    
    # Apply dark theme styling to match Slate theme
    bar_fig.update_layout(
        template='plotly_dark',
        paper_bgcolor='#272b30',
        plot_bgcolor='#272b30',
        font=dict(color='#ffffff')
    )
    
    # Return both updated figures
    return line_fig, bar_fig

# ============================================================================
# RUN APPLICATION
# ============================================================================
# Launch the dashboard on localhost:8057
# debug=True enables auto-reload on code changes and detailed error messages
if __name__ == '__main__':
    app.run(debug=True, port=8057)